# DSA 4060 – Practical Quiz: Personalized Movie Recommender

## Student Details
- **Name:** Claire Mwarari
- **Student ID:** 669470
- **Assigned User ID:** 31

**Assigned User ID calculation:** Last two digits of Student ID = 70;
70 MOD 40 = 30; Assigned User ID = 30 + 1 = **31**.

This notebook analyses the ratings of User 31, profiles their preferences,
and builds a content-based recommender that suggests five movies they have
not yet rated.

## Task 1 – Load and Inspect the Data (3 marks)

I load both CSV files into pandas DataFrames and checked their structure:
first rows, dimensions, and missing values.

In [1]:
import pandas as pd

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

### 1.1 First five rows of each dataset (1 mark)

In [2]:
movies.head()

,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


In [3]:
ratings.head()

,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


### 1.2 Number of rows and columns in each dataset (1 mark)

In [4]:
print('movies.csv shape:', movies.shape)   # (rows, columns)
print('ratings.csv shape:', ratings.shape)

movies.csv shape: (36, 3)
ratings.csv shape: (440, 3)


### 1.3 Missing-value check (1 mark)

In [5]:
print('Missing values in movies.csv:')
print(movies.isna().sum())
print()
print('Missing values in ratings.csv:')
print(ratings.isna().sum())

Missing values in movies.csv:
movie_id    0
title       0
genres      0
dtype: int64

Missing values in ratings.csv:
user_id     0
movie_id    0
rating      0
dtype: int64


**Finding:** There are **no missing values** in either dataset since every
movie has an ID, title and genre string, and every rating has a user ID,
movie ID and rating value. The data is clean and ready to use without any
imputation or row removal.

## Task 2 – Profile the Assigned User (5 marks)

I filtered `ratings.csv` to only the assigned user (User 31) and joined with the
movie catalogue so each rating carries the movie's title and genres.

In [6]:
ASSIGNED_USER = 31   # (70 MOD 40) + 1

user_ratings = ratings[ratings['user_id'] == ASSIGNED_USER]
user_movies = user_ratings.merge(movies, on='movie_id')
print(f'User {ASSIGNED_USER} has rated {len(user_movies)} movies.')

User 31 has rated 10 movies.


### 2.1 All movies rated by the user, with title, genres and rating (2 marks)

In [7]:
user_movies[['movie_id', 'title', 'genres', 'rating']].sort_values(
    'rating', ascending=False).reset_index(drop=True)

,movie_id,title,genres,rating
0,30,The Grand Budapest Hotel,Comedy|Drama,5.0
1,28,Knives Out,Mystery|Comedy,4.0
2,35,Moneyball,Sports|Drama,4.0
3,14,The Notebook,Romance|Drama,4.0
4,32,Jumanji: Welcome to the Jungle,Comedy|Adventure,3.5
5,2,Interstellar,Sci-Fi|Drama,3.0
6,9,Mad Max: Fury Road,Action|Adventure,3.0
7,31,The Hangover,Comedy,2.5
8,3,The Matrix,Sci-Fi|Action,2.0
9,11,Avengers: Endgame,Action|Superhero,2.0


### 2.2 Three highest-rated movies (1 mark)

Ties are broken arbitrarily; any three of the tied highest ratings are valid.

In [8]:
top3 = user_movies.sort_values('rating', ascending=False).head(3)
top3[['title', 'genres', 'rating']].reset_index(drop=True)

,title,genres,rating
0,The Grand Budapest Hotel,Comedy|Drama,5.0
1,Knives Out,Mystery|Comedy,4.0
2,Moneyball,Sports|Drama,4.0


### 2.3 Apparent preferred genre(s), with evidence (2 marks)

In [9]:
# Count how many times each genre appears across the user's ratings,
# split by whether the rating was high (>=4) or low (<4)
from collections import Counter

high, low = Counter(), Counter()
for _, row in user_movies.iterrows():
    for g in row['genres'].split('|'):
        (high if row['rating'] >= 4 else low)[g] += 1

print('Genres in movies rated >= 4:', dict(high))
print('Genres in movies rated < 4 :', dict(low))

Genres in movies rated >= 4: {'Romance': 1, 'Drama': 3, 'Mystery': 1, 'Comedy': 2, 'Sports': 1}
Genres in movies rated < 4 : {'Sci-Fi': 2, 'Drama': 1, 'Action': 3, 'Adventure': 2, 'Superhero': 1, 'Comedy': 2}


**Reasoning:** User 31's apparent preferred genres are **Drama and
Comedy**. Four of their ten ratings are 4.0 or higher — *The Grand Budapest
Hotel* (Comedy|Drama, 5★), *Knives Out* (Mystery|Comedy, 4★), *Moneyball*
(Sports|Drama, 4★) and *The Notebook* (Romance|Drama, 4★) — and **every one
of these liked movies contains Drama, and two also contain Comedy**. In
contrast, their lowest ratings went to action/sci-fi/superhero titles
(*The Matrix* 2★, *Avengers: Endgame* 2★, *Mad Max* 3★, *Interstellar* 3★),
which contain neither Drama nor Comedy. Even *The Hangover* (Comedy) was rated
a mediocre 2.5★, suggesting the user likes comedy mainly when it is paired
with drama or mystery rather than pure comedy. The evidence therefore points
clearly to Drama (in 4/4 liked movies) and Comedy (in 2/4 liked movies) as
the strongest preferences.

## Task 3 – Recommendation Logic (7 marks)

Approach: treat movies rated **4.0 or higher** as evidence of preference,
convert genre strings into numerical features with **TF-IDF**, build a
**centroid profile** of the liked movies, and rank all unseen movies by
**cosine similarity** to that profile.

### 3.1 Select positively rated movies (rating >= 4.0) (1 mark)

In [10]:
liked = user_movies[user_movies['rating'] >= 4.0]
print(f'{len(liked)} movies rated 4.0 or higher:')
liked[['title', 'genres', 'rating']].reset_index(drop=True)

4 movies rated 4.0 or higher:


,title,genres,rating
0,The Notebook,Romance|Drama,4.0
1,Knives Out,Mystery|Comedy,4.0
2,The Grand Budapest Hotel,Comedy|Drama,5.0
3,Moneyball,Sports|Drama,4.0


### 3.2 Vectorise the genres with TF-IDF (2 marks)

The genres column holds pipe-separated strings (e.g. `Sci-Fi|Thriller`).
`TfidfVectorizer` with a custom token pattern splits on the `|` character so
each genre becomes a feature. TF-IDF weights genres by how distinctive they
are across the catalogue — a rare genre like *Sports* carries more weight
than a ubiquitous one like *Drama*.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(token_pattern=r'[^|]+')   # split genres on '|'
tfidf_matrix = vectorizer.fit_transform(movies['genres'])

print('Feature names:', vectorizer.get_feature_names_out())
print('TF-IDF matrix shape:', tfidf_matrix.shape)

Feature names: ['action' 'adventure' 'animation' 'biography' 'comedy' 'crime' 'drama'
 'family' 'horror' 'musical' 'mystery' 'romance' 'sci-fi' 'sports'
 'superhero' 'thriller']
TF-IDF matrix shape: (36, 16)


### 3.3 Compute cosine similarity to the liked-movie profile (2 marks)

I averaged the TF-IDF vectors of the liked movies to form a single
"preference centroid", then computed the cosine similarity between this
centroid and every movie in the catalogue. Cosine similarity ranges from 0
(completely different genres) to 1 (identical genre profile).

In [12]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

liked_rows = movies.index[movies['movie_id'].isin(liked['movie_id'])].tolist()
profile = np.asarray(tfidf_matrix[liked_rows].mean(axis=0))   # centroid of liked movies
similarity = cosine_similarity(profile, tfidf_matrix).flatten()

movies_scored = movies.copy()
movies_scored['similarity'] = similarity
movies_scored[['title', 'genres', 'similarity']].sort_values(
    'similarity', ascending=False).head(8).reset_index(drop=True)

,title,genres,similarity
0,The Grand Budapest Hotel,Comedy|Drama,0.810655
1,The Shawshank Redemption,Drama,0.633419
2,Titanic,Romance|Drama,0.612661
3,The Notebook,Romance|Drama,0.612661
4,Creed,Sports|Drama,0.612661
5,Moneyball,Sports|Drama,0.612661
6,Ford v Ferrari,Sports|Drama,0.612661
7,Remember the Titans,Sports|Drama,0.612661


### 3.4 Exclude already-rated movies and return exactly five (2 marks)

In [13]:
already_rated = user_movies['movie_id']

candidates = movies_scored[~movies_scored['movie_id'].isin(already_rated)]
top5 = candidates.sort_values('similarity', ascending=False).head(5)
print('Check: none of the recommendations were rated by User 31 ->',
      not top5['movie_id'].isin(already_rated).any())
top5[['movie_id', 'title', 'genres', 'similarity']].reset_index(drop=True)

Check: none of the recommendations were rated by User 31 -> True


,movie_id,title,genres,similarity
0,18,The Shawshank Redemption,Drama,0.633419
1,13,Titanic,Romance|Drama,0.612661
2,34,Creed,Sports|Drama,0.612661
3,36,Remember the Titans,Sports|Drama,0.612661
4,33,Ford v Ferrari,Sports|Drama,0.612661


## Task 4 – Explain the Recommendations (3 marks)

To give an evidence-based reason for each pick, I found which *positively
rated* movie each recommendation is most similar to.

In [14]:
dense = cosine_similarity(tfidf_matrix, tfidf_matrix)
id2pos = {mid: i for i, mid in enumerate(movies['movie_id'])}

reasons = []
for _, r in top5.iterrows():
    s = dense[id2pos[r['movie_id']]]
    best_liked = max(liked_rows, key=lambda i: s[i])
    reasons.append((
        r['title'], r['genres'], r['similarity'],
        movies.loc[best_liked, 'title'], s[best_liked]))

results = pd.DataFrame(reasons, columns=[
    'Recommended Movie', 'Genres', 'Similarity to profile',
    'Closest liked movie', 'Similarity to liked movie'])
results.insert(0, 'Rank', range(1, 6))
results

,Rank,Recommended Movie,Genres,Similarity to profile,Closest liked movie,Similarity to liked movie
0,1,The Shawshank Redemption,Drama,0.633419,The Grand Budapest Hotel,0.581095
1,2,Titanic,Romance|Drama,0.612661,The Notebook,1.000000
2,3,Creed,Sports|Drama,0.612661,Moneyball,1.000000
3,4,Remember the Titans,Sports|Drama,0.612661,Moneyball,1.000000
4,5,Ford v Ferrari,Sports|Drama,0.612661,Moneyball,1.000000


**Recommendation table with reasons:**

| Rank | Recommended Movie | Genres | Reason |
|---:|---|---|---|
| 1 | The Shawshank Redemption | Drama | Closest unseen match to the liked profile (0.633); shares the Drama genre present in all four movies User 31 rated ≥ 4★. |
| 2 | Titanic | Romance\|Drama | Identical genre profile (similarity 1.00) to *The Notebook* (4★), the user's liked Romance\|Drama film. |
| 3 | Creed | Sports\|Drama | Identical genre profile (1.00) to *Moneyball* (4★), so it directly extends the user's taste for sports dramas. |
| 4 | Remember the Titans | Sports\|Drama | Same as above — an unseen twin of *Moneyball* (1.00), ranked slightly lower on the profile-centroid tie-break. |
| 5 | Ford v Ferrari | Sports\|Drama | Another exact *Moneyball* genre match (1.00), reinforcing that Sports\|Drama is a strong, reliable signal for this user. |

Note that ranks 2–5 tie at similarity 1.00 to a liked movie; their ordering
comes from the centroid similarity, which blends all four liked films.

## Task 5 – Critical Thinking (2 marks)

**One limitation of this recommender (1 mark):**
It relies **only on genre strings** — five words at most per movie. It ignores
actors, directors, plot, release year and, crucially, *how strongly* the user
preferred each liked movie (a 4★ and a 5★ count equally). As a result it can
only recommend genre clones: four of the five recommendations are Drama or
Sports|Drama variants, so the user gets a narrow filter-bubble list with no
surprise or diversity.

**One realistic improvement (1 mark):**
**Combine the content signal with collaborative filtering** (e.g. a hybrid
model, or matrix factorization such as SVD/ALS on the full 440-rating
matrix). With 40 users, items liked by users with similar taste to User 31
would surface even when their genres differ, capturing factors that genre
strings miss and making recommendations both more personalized and more
diverse. A simpler complementary step would be to **weight liked movies by
their rating** (e.g. a 5★ film contributes more to the centroid than a 4★
film) so the strength of preference, not just its direction, shapes the
recommendations.

## Conclusion

Using only the ratings of assigned User 31, we found a clear Drama/Comedy
preference, built a TF-IDF + cosine-similarity content model over the four
movies rated ≥ 4★, and recommended five unseen movies — *The Shawshank
Redemption*, *Titanic*, *Creed*, *Remember the Titans* and *Ford v Ferrari* —
each justified by its similarity to a movie the user demonstrably liked.